## IMPORTATION DES BIBLIOTHEQUES ET PARAMETRES GLOBAUX

In [1]:
# 1. Imports
import os
import time
import math
import importlib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
from pyproj import CRS

# Recharger le module de fonctions annexes
import fonctions_annexes_biodiv
importlib.reload(fonctions_annexes_biodiv)
from fonctions_annexes_biodiv import generer_dictionnaire_taxonomie

# 2. Constantes et variables globales
%matplotlib qt
path_data = r'D:\MANTIS\Data'



In [2]:
# Fonctions d'importation

def load_geospatial_data(path_data):
    """
    Charge les fichiers SIG globaux et spécifiques à la France.

    Args:
        path_data (str): Chemin du répertoire contenant les fichiers SIG.

    Returns:
        dict: Un dictionnaire contenant les GeoDataFrames chargés.
    """
    sig_path = os.path.join(path_data, "SIG_global")
    france_path = os.path.join(path_data,'GBIF', "GBIF_France", "SIG")

    geodata = {
        "world_terrestre": gpd.read_file(os.path.join(sig_path, "world-administrative-boundaries.geojson")),
        "world_maritime": gpd.read_file(os.path.join(sig_path, "eez_v11.gpkg")),
        "departement_gpd": gpd.read_file(os.path.join(france_path, "carte_departements.geojson")),
        "PNR_gpd": gpd.read_file(os.path.join(france_path, "N_ENP_PNR_S_000.shx"))[['NOM_SITE', 'geometry']],
        "PN_gpd": gpd.read_file(os.path.join(france_path, "N_ENP_PN_S_000.shx"))[['NOM_SITE', 'geometry']]
    }

    return geodata
    
def generate_grid_path(country_name, grid_size_km, grid_type, cle_geo):
    return os.path.join(path_data,'GBIF', f"GBIF_{country_name.replace(' ', '_')}", "SIG",f"grid_{country_name.replace(' ', '_')}_{grid_type}_{cle_geo}.geojson")
      
# Vérifier si les fichiers grid existent et afficher un message approprié
def load_grid_file(path, name):
    if os.path.exists(path):
        print(f"✅ {name} trouvée et chargée avec succès !")
        return gpd.read_file(path)
    else:
        print(f"⚠️ {name} non trouvée. Elle ne sera pas traitée.")
        return None

def charger_donnees_geo(path_data):
    """Charge les données géographiques et retourne les DataFrames pertinents."""
    geo_data = load_geospatial_data(path_data)
    world_terrestre = geo_data["world_terrestre"]
    world_maritime = geo_data["world_maritime"]
    return world_terrestre, world_maritime

    


In [3]:
#Fonctions de génération de la grille

def degrees_per_km(latitude):
    """Calcule la conversion degrés/km pour une latitude donnée."""
    lat_deg_per_km = 1 / 111.32
    lon_deg_per_km = 1 / (111.32 * math.cos(math.radians(latitude)))
    return lat_deg_per_km, lon_deg_per_km

def create_country_grid_WGS84(gdf, country_code, col_code="color_code", grid_size_km=10, midpoint_lat=None, crop=False, display=False):
    """Génère une grille pour un pays spécifique en WGS84."""
    country = gdf[gdf[col_code] == country_code]
    if country.empty:
        raise ValueError(f"Pays '{country_code}' introuvable dans le GeoDataFrame.")

    def create_grid(country, grid_size, reference_point=(0, 0), midpoint_lat=None, crop=False):
        """Création d'une grille avec une taille définie."""
        ref_x, ref_y = reference_point
        country_geometry = country.geometry.union_all()
        minx, miny, maxx, maxy = country_geometry.bounds
        
        if midpoint_lat is None:
            midpoint_lat = (miny + maxy) / 2
        lat_deg_per_km, lon_deg_per_km = degrees_per_km(midpoint_lat)
        dy, dx = grid_size * lat_deg_per_km, grid_size * lon_deg_per_km
        start_x, start_y = ref_x + ((minx - ref_x) // dx) * dx, ref_y + ((miny - ref_y) // dy) * dy
        
        grid_cells = []
        for x in np.arange(start_x, maxx, dx):
            for y in np.arange(start_y, maxy, dy):
                cell = box(x, y, x + dx, y + dy)
                grid_cells.append({"geometry": cell, "min_lon": x, "min_lat": y, "max_lon": x + dx, "max_lat": y + dy})
        
        grid_gdf = gpd.GeoDataFrame(grid_cells, crs=country.crs)
        return gpd.clip(grid_gdf, country_geometry) if crop else grid_gdf
                
    grid = create_grid(country, grid_size_km, midpoint_lat=midpoint_lat, crop=crop)
    grid = grid[grid.intersects(country.geometry.union_all())]
    grid["cell_name"] = grid.apply(lambda cell: f"{grid_size_km}kmE{int(abs(cell.geometry.centroid.x) * 100):05d}N{int(abs(cell.geometry.centroid.y) * 100):05d}{country_code}", axis=1)
    grid["country_code"] = country_code

    if display:
        fig, ax = plt.subplots(figsize=(10, 10))
        gdf.plot(ax=ax, edgecolor="black", linewidth=0.5)
        country.plot(ax=ax, edgecolor="red", linewidth=2, facecolor="none")
        grid.plot(ax=ax, color="lightblue", edgecolor="grey", alpha=0.6)
        plt.show()

    return grid

def create_and_rename_grid(source_gdf, country_code, column_name, grid_size_km, cle_geo, crop,display):
    grid = create_country_grid_WGS84(source_gdf, country_code, column_name, grid_size_km, midpoint_lat=None, crop=crop, display=display)
    return grid.rename(columns={'cell_name': cle_geo})

def check_duplicates(grid, grid_name, cle_geo):
    if not grid.empty:
        duplicate_count = grid[cle_geo].duplicated().sum()
        if duplicate_count > 0:
            print(f"⚠️ Attention : {duplicate_count} doublon(s) trouvé(s) dans {grid_name} !")
        else:
            print(f"✅ Aucun doublon trouvé dans {grid_name}.")

def make_geo_keys_unique(grid, cle_geo):
    if not grid.empty:
        grid["cle_geo_unique"] = grid.groupby(cle_geo).cumcount().astype(str)
        grid.loc[grid["cle_geo_unique"] != "0", cle_geo] += "_" + grid["cle_geo_unique"]
        grid.drop(columns=["cle_geo_unique"], inplace=True)  # Supprime la colonne temporaire

def generer_grille_pays(country_name,grid_size_km,world_terrestre,world_maritime,critere_terrestre="color_code",critere_maritime="ISO_TER1"):
    cle_geo = f"codeMaille{grid_size_km}Km"
    country_code=world_terrestre[world_terrestre["name"] == country_name]["color_code"].iloc[0]
    # 1. Sélection du pays terrestre et maritime
    country_terrestre = world_terrestre[world_terrestre[critere_terrestre] == country_code]
    if country_name == "France":
        country_terrestre = world_terrestre[world_terrestre["iso_3166_1_alpha_2_codes"] == "FR"]
    country_maritime = world_maritime[world_maritime[critere_maritime] == country_code]
    
    # Fusion des géométries terrestres et maritimes
    geom_terrestre = country_terrestre.geometry.union_all()
    geom_maritime = country_maritime.geometry.union_all() if not country_maritime.empty else None
    
    if geom_maritime is not None:
        geom_fusionnee = geom_terrestre.union(geom_maritime)
    else:
        geom_fusionnee = geom_terrestre
    
    # Calcul du centre de latitude pour ajuster la grille
    minx, miny, maxx, maxy = geom_terrestre.bounds
    midpoint_lat = (miny + maxy) / 2
    
    # 3. Création des grilles terrestre, maritime et combinée
    country_grid_terrestre = create_and_rename_grid(country_terrestre, country_code, critere_terrestre, grid_size_km, cle_geo, crop=True,display=False)
    country_grid_maritime = gpd.GeoDataFrame()
    
    if not country_maritime.empty:
        country_grid_maritime = create_and_rename_grid(country_maritime, country_code, critere_maritime, grid_size_km, cle_geo, crop=True,display=False)
    
    # Création d'un GeoDataFrame pour la géométrie fusionnée
    gdf_fusionne = gpd.GeoDataFrame(geometry=[geom_fusionnee], crs=world_terrestre.crs)
    gdf_fusionne["Code"] = country_code
    
    # Création de la grille combinée
    country_grid_combined = create_and_rename_grid(gdf_fusionne, country_code, "Code", grid_size_km, cle_geo, crop=False,display=False)
    
    #Rendre les nom de mailles uniques
    make_geo_keys_unique(country_grid_terrestre, cle_geo)
    make_geo_keys_unique(country_grid_maritime, cle_geo)
    make_geo_keys_unique(country_grid_combined, cle_geo)
    
    #Vérification s'il y a des doublons
    check_duplicates(country_grid_terrestre, "country_grid_terrestre", cle_geo)
    check_duplicates(country_grid_maritime, "country_grid_maritime", cle_geo)
    check_duplicates(country_grid_combined, "country_grid_combined", cle_geo)
    
    # 5. Sauvegarde des fichiers
    country_grid_terrestre.to_file(generate_grid_path(country_name, grid_size_km, 'terrestre', cle_geo), driver="GeoJSON")
    
    if not country_grid_maritime.empty:
        country_grid_maritime.to_file(generate_grid_path(country_name, grid_size_km, 'maritime', cle_geo), driver="GeoJSON")
    
    country_grid_combined.to_file(generate_grid_path(country_name, grid_size_km, 'combined', cle_geo), driver="GeoJSON")


def creer_grille_pays(country_name, grid_size_km, world_terrestre, world_maritime, path_data, cle_geo):
    """Crée la grille pour un pays spécifique et charge les fichiers associés."""
    country_code = world_terrestre[world_terrestre["name"] == country_name]["color_code"].iloc[0]
    generer_grille_pays(country_name, grid_size_km, world_terrestre, world_maritime, critere_terrestre="color_code", critere_maritime="ISO_TER1")

    path_grid_terrestre = generate_grid_path(country_name, grid_size_km, 'terrestre', cle_geo)
    path_grid_maritime = generate_grid_path(country_name, grid_size_km, 'maritime', cle_geo)
    path_grid_combined = generate_grid_path(country_name, grid_size_km, 'combined', cle_geo)

    country_grid_terrestre = load_grid_file(path_grid_terrestre, "Grid Terrestre")
    country_grid_maritime = load_grid_file(path_grid_maritime, "Grid Maritime")
    country_grid_combined = load_grid_file(path_grid_combined, "Grid Combined")

    return country_grid_terrestre, country_grid_maritime, country_grid_combined


In [4]:
def process_biodiv_data(df,annee_min=1):
    """Nettoie et formate les données de biodiversité."""
    
    # Copier les données pour éviter les modifications sur l'original
    df_cleaned = df.copy()
    
    # Identifier le nombre initial d'espèces et d'observations
    n_especes_entrée = len(df_cleaned[cle_ID].unique())
    n_obs_entrée = len(df_cleaned)
    
    # 🔹 Convertir 'eventDate' en datetime et compléter 'year'
    df_cleaned['eventDate'] = pd.to_datetime(df_cleaned['eventDate'], errors='coerce', utc=True)
    df_cleaned['year'] = df_cleaned['year'].fillna(df_cleaned['eventDate'].dt.year)
    
    # 🔹 Assurez-vous que les coordonnées sont numériques et filtrer les NaN
    df_cleaned['decimalLongitude'] = pd.to_numeric(df_cleaned['decimalLongitude'], errors='coerce')
    df_cleaned['decimalLatitude'] = pd.to_numeric(df_cleaned['decimalLatitude'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['decimalLongitude', 'decimalLatitude', cle_ID]).reset_index(drop=True)

    print(f"➡️  En entrée : {n_especes_entrée} espèces, {n_obs_entrée} observations")

    # Suppression des lignes avec valeurs manquantes pour les colonnes cruciales
    df_cleaned = df_cleaned.dropna(subset=[cle_ID]).reset_index(drop=True)

    # Filtrer les observations où 'occurrenceStatus' est 'PRESENT'
    df_cleaned = df_cleaned[df_cleaned['occurrenceStatus'] == 'PRESENT'].reset_index(drop=True)

    # Convertir l'ID des espèces en entier
    df_cleaned[cle_ID] = df_cleaned[cle_ID].astype(int)
    
    df_cleaned['year'] = pd.to_numeric(df_cleaned['year'], errors='coerce').fillna(0).astype(int)
    if annee_min is not None:
        df_cleaned = df_cleaned[df_cleaned['year'] >= annee_min]

    # Renommer la colonne contenant la maille géographique
    df_cleaned.rename(columns={'grid_name': cle_geo}, inplace=True)

    # Convertir 'individualCount' en numérique et remplacer NaN par 1
    df_cleaned['individualCount'] = pd.to_numeric(df_cleaned['individualCount'], errors='coerce').fillna(1)

    # Calcul des pertes en pourcentage
    perte_especes = 100 - round(len(df_cleaned[cle_ID].unique()) / n_especes_entrée * 100)
    perte_obs = 100 - round(len(df_cleaned) / n_obs_entrée * 100)

    print(f"✅ En sortie : {len(df_cleaned[cle_ID].unique())} espèces (-{perte_especes}%)")
    print(f"✅ En sortie : {len(df_cleaned)} observations (-{perte_obs}%)")

    return df_cleaned

def add_grid_to_country(df_country, grid,cle_geo):
    df_new=df_country.copy()
    """
    Optimized version of adding the corresponding grid cell to each row in df_country based on latitude and longitude.
    
    Parameters:
    - df_country: DataFrame containing the columns 'decimalLatitude' and 'decimalLongitude'.
    - grid: DataFrame containing the grid cells with 'name', 'min_lon', 'min_lat', 'max_lon', and 'max_lat' columns.
    
    Returns:
    - df_country: Updated DataFrame with an additional 'grid_name' column indicating the grid cell for each point.
    """
    """
    # Convert grid bounds to NumPy arrays for efficient vectorized comparison
    min_lons = grid['min_lon'].values
    max_lons = grid['max_lon'].values
    min_lats = grid['min_lat'].values
    max_lats = grid['max_lat'].values
    """
    
    # Assure-toi que grid['geometry'] contient des Polygons
    bounds = grid['geometry'].bounds  # Cela renvoie un DataFrame avec minx, miny, maxx, maxy
    
    min_lons = bounds['minx'].values
    max_lons = bounds['maxx'].values
    min_lats = bounds['miny'].values
    max_lats = bounds['maxy'].values

    
    grid_names = grid[cle_geo].values


    # Initialize an array to store the grid names
    grid_names_for_points = []
    s=0
    n=0
    print(f"➡️ Association des mailles aux données")
    # Iterate over each point in df_country and apply vectorized comparison
    for lon, lat in zip(df_country['decimalLongitude'], df_country['decimalLatitude']):
        # Find the grid cell by comparing the point coordinates with grid bounds
        matching_grid = np.where((min_lons <= lon) & (lon <= max_lons) & (min_lats <= lat) & (lat <= max_lats))[0]
        
        if matching_grid.size > 0:
            grid_names_for_points.append(grid_names[matching_grid[0]])  # Take the first matching grid cell
            if matching_grid.size > 1:
                s=s+1
      
        else:
            grid_names_for_points.append(None)  # No matching grid
            n=n+1
       
    print(f'{s} with several matching grids')
    print(f'{n} with no matching grid')
    # Add the grid names to the DataFrame
    df_new[cle_geo] = grid_names_for_points
    
    return df_new
    
def formater_maille_espece_GBIF(df,cle_geo='codeMaille10Km',cle_ID='cdRef',bornes_temporelles=None):
    df_dico=generer_dictionnaire_taxonomie(df,cle_ID)
    # Convertir la colonne 'year' en int
    df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)

    # Choisir des bornes temporelles et assigner une période aux données
    if bornes_temporelles is not None:
        df.loc[:, 'periode'] = pd.cut(df['year'], bins=bornes_temporelles, 
                       labels=[f'Période {i+1}: {bornes_temporelles[i]+1} à {bornes_temporelles[i+1]}' for i in range(len(bornes_temporelles) - 1)],
                       include_lowest=False)  # include_lowest=True inclut la borne inférieureure
        # Compter le nombre de données dans chaque intervalle
        compte_par_periode = df['periode'].value_counts()

         # Compter les occurrences d'observation de chaque taxon pour chaque code et période
        df_maille_espece = df.groupby([cle_geo, cle_ID,'periode'], observed=True).size().reset_index(name='nombreObs')
        #df_maille_espece = df.groupby([cle_geo, cle_ID,'periode'], observed=True)['individualCount'].sum().reset_index(name='nombreObs')
        #eventuellement remplacer ['individualCount'].sum() par .size() 
       
    else:
        # Compter les occurrences d'observation de chaque taxon pour chaque code
        df_maille_espece = df.groupby([cle_geo, cle_ID], observed=True).size().reset_index(name='nombreObs')
        #eventuellement remplacer ['individualCount'].sum() par .size() 
        
    df_maille_espece=pd.merge(df_maille_espece,df_dico,on=cle_ID)
    
    return df_maille_espece
    
# 📌 Fonction pour charger et traiter chaque chunk
def process_chunk(df_biodiv, country_grid_terrestre, country_grid_maritime, country_grid_combined, bornes_temporelles, chunk_number):
    """Prétraitement et enregistrement des données par chunk"""
    
    print(f"\n🔍 Traitement du chunk {chunk_number}...")

    df_biodiv = process_biodiv_data(df_biodiv)

    # 🔹 Ajouter la maille pour chaque catégorie
    if country_grid_terrestre is not None:
        df_terrestre = add_grid_to_country(df_biodiv, country_grid_terrestre, cle_geo).dropna(subset=[cle_geo]).reset_index(drop=True)
        
    if country_grid_maritime is not None:
        df_maritime = add_grid_to_country(df_biodiv, country_grid_maritime, cle_geo).dropna(subset=[cle_geo]).reset_index(drop=True)

    if country_grid_combined is not None:
        df_combined = add_grid_to_country(df_biodiv, country_grid_combined, cle_geo).dropna(subset=[cle_geo]).reset_index(drop=True)

    # 🔹 Regrouper par maille et période
    chaine_bornes = "_".join(map(str, bornes_temporelles))
    
    if country_grid_terrestre is not None:
        df_maille_espece_terrestre = formater_maille_espece_GBIF(df_terrestre, cle_geo, cle_ID, bornes_temporelles)
    if country_grid_maritime is not None:
        df_maille_espece_maritime = formater_maille_espece_GBIF(df_maritime, cle_geo, cle_ID, bornes_temporelles)
    if country_grid_combined is not None:
        df_maille_espece_combined = formater_maille_espece_GBIF(df_combined, cle_geo, cle_ID, bornes_temporelles)

    # 🔹 Ajouter les noms vernaculaires
    dico_noms_vernaculaires = pd.read_csv(path_data + r"\TAXO_GBIF\dico_noms_vernaculaires_merged.csv")

    if country_grid_terrestre is not None:
        df_maille_espece_terrestre = pd.merge(df_maille_espece_terrestre, dico_noms_vernaculaires, on=cle_ID, how="left")
    if country_grid_maritime is not None:
        df_maille_espece_maritime = pd.merge(df_maille_espece_maritime, dico_noms_vernaculaires, on=cle_ID, how="left")
    if country_grid_combined is not None:
        df_maille_espece_combined = pd.merge(df_maille_espece_combined, dico_noms_vernaculaires, on=cle_ID, how="left")
    
    # 🔹 Sauvegarde des fichiers
    
    def save_chunk(df, ecosysteme,chunk_number):
        
        # Créer le répertoire s'il n'existe pas
        path_save=os.path.join(path_data,'GBIF',f"GBIF_{country_name.replace(' ', '_')}", 'processed')
        os.makedirs(path_save, exist_ok=True)
        df.to_csv(os.path.join(path_save, f"data_GBIF_{country_name.replace(' ', '_')}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}_{chunk_number}.csv"), index=False)

    if country_grid_terrestre is not None:
        save_chunk(df_maille_espece_terrestre, "terrestre",chunk_number)
    if country_grid_maritime is not None:
        save_chunk(df_maille_espece_maritime, "maritime",chunk_number)
    if country_grid_combined is not None:
        save_chunk(df_maille_espece_combined, "combined",chunk_number)

    # 🔹 Nettoyage mémoire
# 🔹 Nettoyage mémoire
if 'df_biodiv' in locals():
    del df_biodiv
if 'df_terrestre' in locals():
    del df_terrestre
if 'df_maritime' in locals():
    del df_maritime
if 'df_combined' in locals():
    del df_combined
if 'df_maille_espece_terrestre' in locals():
    del df_maille_espece_terrestre
if 'df_maille_espece_maritime' in locals():
    del df_maille_espece_maritime
if 'df_maille_espece_combined' in locals():
    del df_maille_espece_combined

    print(f"\n✅ Chunk {chunk_number} traité et sauvegardé avec succès ! 🎉\n")

def traiter_chunks(path_fichier, colonnes_a_importer, country_grid_terrestre, country_grid_maritime, country_grid_combined, bornes_temporelles):
    """Lit les données par chunks et les traite pour chaque écosystème."""
    chunk_number = 0
    import csv
    for df_biodiv in pd.read_csv(path_fichier, sep='\t', encoding='utf-8', quoting=csv.QUOTE_NONE, chunksize=10_000_000, on_bad_lines='skip', usecols=colonnes_a_importer):
        chunk_number += 1
        process_chunk(df_biodiv, country_grid_terrestre, country_grid_maritime, country_grid_combined, bornes_temporelles, chunk_number)
    return chunk_number

def fusionner_fichiers_par_ecosysteme(country_name, chunk_number, ecosysteme, cle_geo, cle_ID, bornes_temporelles, path_data):
    """Fusionne les fichiers pour un écosystème donné et génère le fichier final."""
    chaine_bornes = "_".join(map(str, bornes_temporelles))
    df_final = pd.DataFrame()
    fichiers_trouves = False 

    for i in range(1, chunk_number + 1):
        file_path = os.path.join(path_data,'GBIF',f"GBIF_{country_name.replace(' ', '_')}", 'processed', f"data_GBIF_{country_name.replace(' ', '_')}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}_{i}.csv")
        if os.path.exists(file_path):
            fichiers_trouves = True
            df_temp = pd.read_csv(file_path)
            df_final = pd.concat([df_final, df_temp], ignore_index=True)
    # Si aucun fichier n'a été trouvé, on arrête la fonction
    if not fichiers_trouves:
        print(f"⚠️ Aucun fichier trouvé pour {country_name} - {ecosysteme} - {cle_geo}. Aucune fusion effectuée.")
        return None

    # Regroupement des observations
    df_maille_espece = df_final.groupby([cle_geo, cle_ID, 'periode'], observed=True)['nombreObs'].sum().reset_index()

    # Génération du dictionnaire taxonomique
    df_dico = generer_dictionnaire_taxonomie(df_final, cle_ID)

    # Fusion des données
    df_maille_espece = pd.merge(df_maille_espece, df_dico, on=cle_ID)

    # Sauvegarde du fichier fusionné
    final_file_path = os.path.join(path_data,'GBIF',f"GBIF_{country_name.replace(' ', '_')}", 'processed', f"data_GBIF_{country_name.replace(' ', '_')}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}.csv")
    df_maille_espece.to_csv(final_file_path, index=False)

    # Suppression des fichiers chunk après fusion
    for i in range(1, chunk_number + 1):
        file_path = os.path.join(path_data,'GBIF',f"GBIF_{country_name.replace(' ', '_')}", 'processed', f"data_GBIF_{country_name.replace(' ', '_')}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}_{i}.csv")
        if os.path.exists(file_path):
            os.remove(file_path)

    return final_file_path



In [5]:
def traiter_pays_et_maille(country_name, grid_size_km, bornes_temporelles, path_data, cle_geo, cle_ID):
    """Exécute le traitement complet pour un pays et une taille de maille donnés."""
    world_terrestre, world_maritime = charger_donnees_geo(path_data)
    
    country_grid_terrestre, country_grid_maritime, country_grid_combined = creer_grille_pays(
        country_name, grid_size_km, world_terrestre, world_maritime, path_data, cle_geo
    )
    
    path_fichier = os.path.join(path_data,'GBIF',f"GBIF_{country_name.replace(' ', '_')}", "Raw", f"extractGBIF_{country_name.replace(' ', '_')}_12112024.csv")
    colonnes_a_importer = ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'verbatimScientificName',
                           'taxonRank', 'countryCode', 'occurrenceStatus', 'individualCount', 'decimalLongitude',
                           'decimalLatitude', 'eventDate', 'speciesKey', 'occurrenceID', 'year']
    
    chunk_number = traiter_chunks(path_fichier, colonnes_a_importer, country_grid_terrestre, country_grid_maritime, country_grid_combined, bornes_temporelles)

    processed_path = os.path.join(path_data, "processed")
    os.makedirs(processed_path, exist_ok=True)

    for ecosysteme in ['maritime', 'combined', 'terrestre']:
        fusionner_fichiers_par_ecosysteme(
            country_name, chunk_number, ecosysteme, cle_geo, cle_ID, bornes_temporelles, path_data
        )

    print(f"✅ Traitement complet pour {country_name} avec une taille de maille {grid_size_km} km")



## SCRIPT ENTIER

In [6]:

filtered_countries=['France']
tailles_maille = [5]
cle_ID="speciesKey"

# Bornes temporelles modifiables
bornes_temporelles = [1800, 1990,2010, 2024]  # Vous pouvez les ajuster ici
chaine_bornes = "_".join(map(str, bornes_temporelles))

for country_name in filtered_countries:
    country_name_underscore=country_name.replace('_', ' ')
    for grid_size_km in tailles_maille:
        cle_geo = f"codeMaille{grid_size_km}Km"
        print(f"\n📢 Traitement pour le pays : {country_name_underscore} avec une taille de maille : {grid_size_km} km\n")
        cle_geo = f"codeMaille{grid_size_km}Km"
        traiter_pays_et_maille(country_name_underscore, grid_size_km, bornes_temporelles, path_data, cle_geo, cle_ID)



📢 Traitement pour le pays : France avec une taille de maille : 5 km

✅ Aucun doublon trouvé dans country_grid_terrestre.
✅ Aucun doublon trouvé dans country_grid_maritime.
✅ Aucun doublon trouvé dans country_grid_combined.
✅ Grid Terrestre trouvée et chargée avec succès !
✅ Grid Maritime trouvée et chargée avec succès !
✅ Grid Combined trouvée et chargée avec succès !

🔍 Traitement du chunk 1...
➡️  En entrée : 24988 espèces, 10000000 observations
✅ En sortie : 23519 espèces (-6%)
✅ En sortie : 9439914 observations (-6%)
➡️ Association des mailles aux données
2 with several matching grids
263936 with no matching grid
➡️ Association des mailles aux données
0 with several matching grids
9103294 with no matching grid
➡️ Association des mailles aux données
2 with several matching grids
4457 with no matching grid

🔍 Traitement du chunk 2...
➡️  En entrée : 18888 espèces, 10000000 observations
✅ En sortie : 14427 espèces (-24%)
✅ En sortie : 9679504 observations (-3%)
➡️ Association des mai

C:\Users\User 1\AppData\Local\Temp\ipykernel_13280\2582636191.py:212: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_biodiv in pd.read_csv(path_fichier, sep='\t', encoding='utf-8', quoting=csv.QUOTE_NONE, chunksize=10_000_000, on_bad_lines='skip', usecols=colonnes_a_importer):



🔍 Traitement du chunk 3...
➡️  En entrée : 25787 espèces, 10000000 observations
✅ En sortie : 21744 espèces (-16%)
✅ En sortie : 9630518 observations (-4%)
➡️ Association des mailles aux données
16 with several matching grids
652262 with no matching grid
➡️ Association des mailles aux données
78 with several matching grids
8532541 with no matching grid
➡️ Association des mailles aux données
94 with several matching grids
13898 with no matching grid

🔍 Traitement du chunk 4...
➡️  En entrée : 24604 espèces, 10000000 observations
✅ En sortie : 22771 espèces (-7%)
✅ En sortie : 9940454 observations (-1%)
➡️ Association des mailles aux données
50 with several matching grids
497901 with no matching grid
➡️ Association des mailles aux données
0 with several matching grids
8839100 with no matching grid
➡️ Association des mailles aux données
50 with several matching grids
4531 with no matching grid


C:\Users\User 1\AppData\Local\Temp\ipykernel_13280\2582636191.py:212: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_biodiv in pd.read_csv(path_fichier, sep='\t', encoding='utf-8', quoting=csv.QUOTE_NONE, chunksize=10_000_000, on_bad_lines='skip', usecols=colonnes_a_importer):



🔍 Traitement du chunk 5...
➡️  En entrée : 24187 espèces, 10000000 observations
✅ En sortie : 23759 espèces (-2%)
✅ En sortie : 9627641 observations (-4%)
➡️ Association des mailles aux données
9 with several matching grids
242951 with no matching grid
➡️ Association des mailles aux données
1 with several matching grids
9177868 with no matching grid
➡️ Association des mailles aux données
10 with several matching grids
2797 with no matching grid

🔍 Traitement du chunk 6...
➡️  En entrée : 8165 espèces, 10000000 observations
✅ En sortie : 8164 espèces (-0%)
✅ En sortie : 9989076 observations (-0%)
➡️ Association des mailles aux données
0 with several matching grids
216996 with no matching grid
➡️ Association des mailles aux données
0 with several matching grids
9717889 with no matching grid
➡️ Association des mailles aux données
0 with several matching grids
17056 with no matching grid

🔍 Traitement du chunk 7...
➡️  En entrée : 7657 espèces, 10000000 observations
✅ En sortie : 7655 esp

...


🔍 Traitement du chunk 17...
➡️  En entrée : 44592 espèces, 10000000 observations
✅ En sortie : 32195 espèces (-28%)
✅ En sortie : 8689781 observations (-13%)
➡️ Association des mailles aux données
33 with several matching grids
317100 with no matching grid
➡️ Association des mailles aux données
0 with several matching grids
8119408 with no matching grid
➡️ Association des mailles aux données
33 with several matching grids
7683 with no matching grid


C:\Users\User 1\AppData\Local\Temp\ipykernel_13280\2582636191.py:212: DtypeWarning: Columns (2,5,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_biodiv in pd.read_csv(path_fichier, sep='\t', encoding='utf-8', quoting=csv.QUOTE_NONE, chunksize=10_000_000, on_bad_lines='skip', usecols=colonnes_a_importer):



🔍 Traitement du chunk 18...
➡️  En entrée : 48447 espèces, 10000000 observations
✅ En sortie : 36785 espèces (-24%)
✅ En sortie : 8966610 observations (-10%)
➡️ Association des mailles aux données
43 with several matching grids
467860 with no matching grid
➡️ Association des mailles aux données
15 with several matching grids
8172854 with no matching grid
➡️ Association des mailles aux données
58 with several matching grids
3942 with no matching grid


C:\Users\User 1\AppData\Local\Temp\ipykernel_13280\2582636191.py:212: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_biodiv in pd.read_csv(path_fichier, sep='\t', encoding='utf-8', quoting=csv.QUOTE_NONE, chunksize=10_000_000, on_bad_lines='skip', usecols=colonnes_a_importer):



🔍 Traitement du chunk 19...
➡️  En entrée : 54166 espèces, 10000000 observations
✅ En sortie : 44167 espèces (-18%)
✅ En sortie : 8839270 observations (-12%)
➡️ Association des mailles aux données
45 with several matching grids
299008 with no matching grid
➡️ Association des mailles aux données
6 with several matching grids
8352949 with no matching grid
➡️ Association des mailles aux données
51 with several matching grids
2182 with no matching grid

🔍 Traitement du chunk 20...
➡️  En entrée : 38659 espèces, 4762944 observations
✅ En sortie : 33297 espèces (-14%)
✅ En sortie : 4280299 observations (-10%)
➡️ Association des mailles aux données
3 with several matching grids
141738 with no matching grid
➡️ Association des mailles aux données
0 with several matching grids
4057496 with no matching grid
➡️ Association des mailles aux données
3 with several matching grids
1611 with no matching grid
✅ Traitement complet pour France avec une taille de maille 5 km


In [ ]:
# Chemin du dossier
path_data = r'D:\MANTIS\Data'
# Liste des pays à traiter et tailles de maille
tailles_maille = [50]
cle_ID = "speciesKey"
# Bornes temporelles modifiables
bornes_temporelles = [1800, 1990, 2010, 2024]
exclude=False

# Liste pour stocker les noms de pays
countries = []
excluded_countries=[]
chaine_bornes = "_".join(map(str, bornes_temporelles))

# Parcours des sous-dossiers du dossier
for folder in os.listdir(os.path.join(path_data, 'GBIF')):
    # Vérifie si le nom du dossier correspond à 'GBIF_{country_name}'
    if folder.startswith('GBIF_'):
        country_name = folder.replace('GBIF_', '')  # Extraire le nom du pays
        countries.append(country_name)

if exclude:
    excluded_countries_file = os.path.join(path_data, "excluded_countries.txt")
    # Charger la liste des pays exclus au début
    if os.path.exists(excluded_countries_file):
        with open(excluded_countries_file, "r", encoding="utf-8") as f:
            excluded_countries = [line.strip() for line in f.readlines()]
    else:
        excluded_countries = []
    # Filtrer la liste des pays
    filtered_countries = [country for country in countries if country not in excluded_countries]
else:
    filtered_countries=countries

for country_name in filtered_countries:
    country_name_underscore = country_name.replace('_', ' ')
    try:
        for grid_size_km in tailles_maille:
            cle_geo = f"codeMaille{grid_size_km}Km"
            print(f"\n📢 Traitement pour le pays : {country_name_underscore} avec une taille de maille : {grid_size_km} km")
            traiter_pays_et_maille(country_name_underscore, grid_size_km, bornes_temporelles, path_data, cle_geo, cle_ID)
        
        # Si tout se passe bien, ajouter le pays aux exclus
        excluded_countries.append(country_name)

    except Exception as e:
        print(f"⚠️ Erreur lors du traitement de {country_name_underscore}: {e}")
        # Ne pas ajouter à excluded_countries en cas d'erreur

# Sauvegarder la liste mise à jour des pays exclus
with open(excluded_countries_file, "w", encoding="utf-8") as f:
    for country in excluded_countries:
        f.write(country + "\n")
